# Caracal Bench Full - GPU T4 x2

Roda TODOS os benches (probe + MMLU + HumanEval) em GPU T4 x2.
TPU nao serve pra probe+HumanEval (XLA recompila .generate() por shape).

**Settings:**
1. Accelerator -> GPU T4 x2
2. Internet ON
3. Persistence Variables and Files

Tempo: ~2h adapter + ~1.5h base = ~3.5h total (cabe nos 12h Kaggle).

In [ ]:
CHECKPOINT_DATASET = "pedroafonso2/caracal-base-3b-s01"
OUTPUT_DATASET = "pedroafonso2/caracal-bench-full-s01"
print(f"checkpoint={CHECKPOINT_DATASET} -> GPU bench (probe + MMLU + HumanEval)")

In [ ]:
!pip install -q 'transformers>=4.46.0' 'peft>=0.13.0' 'datasets>=3.0.0' kaggle

In [ ]:
import os
import subprocess

if not os.path.exists("/kaggle/working/caracal-1"):
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "-b",
            "dev",
            "https://github.com/iterate-labs-ai/caracal-1.git",
            "/kaggle/working/caracal-1",
        ],
        check=True,
    )
os.chdir("/kaggle/working/caracal-1")
rev = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip()
print(f"cloned, HEAD={rev}", flush=True)

In [ ]:
import torch

print(f"CUDA: {torch.cuda.is_available()}, n_gpu: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  gpu{i}: {torch.cuda.get_device_name(i)}")

In [ ]:
import subprocess

ckpt_dir = "/kaggle/working/ckpt"
subprocess.run(
    [
        "kaggle",
        "datasets",
        "download",
        "-d",
        CHECKPOINT_DATASET,
        "-p",
        ckpt_dir,
        "--unzip",
        "--force",
    ],
    check=True,
)
subprocess.run(["ls", "-la", ckpt_dir], check=True)

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    "-u",
    "eval/run_bench_all.py",
    "--adapter",
    ckpt_dir,
    "--out-dir",
    "/kaggle/working/bench-adapter",
]
print("Running adapter bench:", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True)

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    "-u",
    "eval/run_bench_all.py",
    "--out-dir",
    "/kaggle/working/bench-base",
]
print("Running base bench:", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True)

In [ ]:
import json
from pathlib import Path

adapter_sum = json.loads(Path("/kaggle/working/bench-adapter/summary.json").read_text())
base_sum = json.loads(Path("/kaggle/working/bench-base/summary.json").read_text())


def metric(s, name, key):
    step = s.get(name, {})
    if step.get("status") != "ok":
        return None
    return step.get("metrics", {}).get(key)


rows = [
    ("probe.mean_ppl", "probe", "mean_ppl"),
    ("probe.cwe_hit_rate", "probe", "cwe_hit_rate"),
    ("mmlu_security.acc", "mmlu_security", "accuracy"),
    ("secqa.acc", "secqa", "accuracy"),
    ("cybermetric.acc", "cybermetric", "accuracy"),
    ("cti_bench.rcm.acc", "cti_bench", "rcm"),
    ("cti_bench.vsp.acc", "cti_bench", "vsp"),
    ("humaneval.pass_at_1", "humaneval", "pass_at_1"),
]
deltas = {}
for label, name, key in rows:
    a = metric(adapter_sum, name, key)
    b = metric(base_sum, name, key)
    if isinstance(a, dict):
        a = a.get("accuracy")
    if isinstance(b, dict):
        b = b.get("accuracy")
    deltas[label] = {
        "adapter": a,
        "base": b,
        "delta": (a - b) if (a is not None and b is not None) else None,
    }

diff = {
    "checkpoint_dataset": CHECKPOINT_DATASET,
    "adapter": adapter_sum,
    "base": base_sum,
    "deltas": deltas,
}
pub_dir = Path("/kaggle/working/bench-published")
pub_dir.mkdir(parents=True, exist_ok=True)
(pub_dir / "bench_full_vs_base.json").write_text(json.dumps(diff, indent=2))
print(json.dumps(deltas, indent=2))

In [ ]:
import json
import subprocess

metadata = {
    "title": "Caracal Bench Full GPU",
    "id": OUTPUT_DATASET,
    "licenses": [{"name": "Apache-2.0"}],
}
(pub_dir / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2))

r = subprocess.run(
    ["kaggle", "datasets", "create", "-p", str(pub_dir), "--public"],
    capture_output=True,
    text=True,
    check=False,
)
print(r.stdout, r.stderr)
if r.returncode != 0:
    subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(pub_dir), "-m", "bench full"], check=True
    )
print(f"Published -> {OUTPUT_DATASET}")